# Uncertainty-guided self-review

A LangGraph agent scores an archived draft, reviews the most uncertain
lines, and proposes a revision. Plain review uses the same prompt and
generation budget with the uncertainty hints omitted.

In the [project environment](README), install the agent dependency.

```powershell
.\.venv\Scripts\python.exe -m pip install langgraph==1.0.10
```
For the Python checks, start Docker and run `docker pull python:3.11-slim`.
Set `check_python = False` to run the reference comparison alone.

In [1]:
from pathlib import Path
import sys

if Path("traces.py").is_file():
    sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import torch
from IPython.display import display

from experiments.agent_loop import run_benchmark
from experiments.data import DATASETS
from experiments.evaluation import TraceStore

torch.set_num_threads(min(4, torch.get_num_threads()))
store = TraceStore(quantize=True)
cases_per_benchmark = 4
check_python = True

Each method fits on that benchmark's original training split. Every
condition starts with the same held-out draft and gets one review round.
The agent sees the task, draft, and up to two ranked lines. The reference
solution and Python assertions are used by the evaluator.

Python cases are sampled from tasks whose canonical solution passes the archived assertions. The run uses a fixed seed.

In [2]:
rows = []
for benchmark in DATASETS:
    rows.extend(
        run_benchmark(
            store,
            benchmark,
            limit=cases_per_benchmark,
            check_python=check_python,
        )
    )
    print(f"Finished {benchmark}")
results = pd.DataFrame(rows)
for benchmark, group in results.groupby("benchmark", sort=False):
    print(benchmark, ", ".join(group["task"].unique()))

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Finished mbpp


Finished humaneval


Finished humaneval_java


Finished defects4j
mbpp 569, 391, 268, 723
humaneval 129, 127, 64, 115
humaneval_java 114, 36, 24, 139
defects4j Closure_52, Compress_30, Closure_35, JacksonDatabind_76


### Held-out results

Similarity compares code tokens with the archived canonical solution.
Exact matches use the same token representation. Python pass counts use
the archived assertions. Java results use the reference comparison.

In [3]:
plain = results.loc[results["method"] == "Plain review"]
baseline = plain.set_index(["benchmark", "task"])["similarity"]
results = results.join(baseline.rename("plain_similarity"), on=["benchmark", "task"])
results["delta_vs_plain"] = results["similarity"] - results["plain_similarity"]

grouped = results.groupby(["benchmark", "method"], sort=False)
summary = grouped.agg(
    tasks=("task", "size"),
    similarity=("similarity", "mean"),
    delta_vs_plain=("delta_vs_plain", "mean"),
    exact=("exact", "sum"),
    valid_replies=("accepted", "sum"),
)
summary["python_passes"] = grouped["passed"].sum(min_count=1).astype("Int64")
display(summary.round(4))

tasks  similarity  delta_vs_plain  exact  \
benchmark      method                                                     
mbpp           Archived draft      4      0.6357         -0.0913      0   
               Plain review        4      0.7269          0.0000      0   
               Lookback Lens       4      0.5893         -0.1377      0   
               UQ Head             4      0.5893         -0.1377      0   
               ICR Probe           4      0.5843         -0.1426      0   
               SAPLMA              4      0.5893         -0.1377      0   
               HaloScope           4      0.5843         -0.1426      0   
humaneval      Archived draft      4      0.4115         -0.1376      0   
               Plain review        4      0.5491          0.0000      0   
               Lookback Lens       4      0.4655         -0.0837      0   
               UQ Head             4      0.5258         -0.0234      0   
               ICR Probe           4      0.6418          0.0927      0   
               SAPLMA              4      0.5313         -0.0178      0   
               HaloScope           4      0.5657          0.0166      0   
humaneval_java Archived draft      4      0.5843          0.0525      0   
               Plain review        4      0.5318          0.0000      0   
               Lookback Lens       4      0.6399          0.1081      0   
               UQ Head             4      0.6399          0.1081      0   
               ICR Probe           4      0.6399          0.1081      0   
               SAPLMA              4      0.5831          0.0512      0   
               HaloScope           4      0.6399          0.1081      0   
defects4j      Archived draft      4      0.2808          0.0355      0   
               Plain review        4      0.2453          0.0000      0   
               Lookback Lens       4      0.3056          0.0603      0   
               UQ Head             4      0.3129          0.0675      0   
               ICR Probe           4      0.4153          0.1700      0   
               SAPLMA              4      0.3107          0.0654      0   
               HaloScope           4      0.3045          0.0592      0   

                               valid_replies  python_passes  
benchmark      method                                        
mbpp           Archived draft              4              0  
               Plain review                4              1  
               Lookback Lens               4              2  
               UQ Head                     4              2  
               ICR Probe                   4              2  
               SAPLMA                      4              2  
               HaloScope                   4              2  
humaneval      Archived draft              4              0  
               Plain review                3              0  
               Lookback Lens               2              1  
               UQ Head                     3              1  
               ICR Probe                   4              1  
               SAPLMA                      3              0  
               HaloScope                   3              0  
humaneval_java Archived draft              4           <NA>  
               Plain review                4           <NA>  
               Lookback Lens               4           <NA>  
               UQ Head                     4           <NA>  
               ICR Probe                   4           <NA>  
               SAPLMA                      4           <NA>  
               HaloScope                   4           <NA>  
defects4j      Archived draft              4           <NA>  
               Plain review                4           <NA>  
               Lookback Lens               4           <NA>  
               UQ Head                     4           <NA>  
               ICR Probe                   4           <NA>  
               SAPLMA                      3

In [4]:
example = results.loc[results["method"] == "Lookback Lens"].iloc[0]
print(f"{example['benchmark']} task {example['task']}")
print("Hints passed to the agent")
for line, excerpt in example["hints"]:
    print(f"{line}: {excerpt}")
print(example["review"])

mbpp task 569
Hints passed to the agent
3: return sorted(list1, key=lambda x: sorted(x))
1: def sort_sublists(list1):
```python
def sort_sublists(list1):
    return sorted(list1, key=lambda x: tuple(sorted(x)))
```

Review: The current draft is almost correct, but it needs a slight modification to ensure that the sorting is stable and the sublists are sorted in-place.

The issue with the current draft is that it sorts the sublists using the `sorted` function, which returns a new sorted list and leaves the original list unchanged. This can be inefficient if the sublists are large. To fix this, we can use the `sorted` function with the `key` argument set to a lambda function that returns a tuple of the sorted sublist. This ensures that the sorting is stable and the sublists are sorted in-place.

Alternatively, you could use the `sorted` function with the `key` argument set to a lambda function that returns the sorted sublist itself, like this:
